## 2. Simulate a small company

a) Connect python to gemini, very important that you place the api key in .env and gitignore it

In [10]:
from google import genai
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")

client = genai.Client(api_key=api_key)

response = client.models.generate_content(
    model="gemini-2.5-flash", contents="Explain how AI works in a few words"
)
print(response.text)

AI learns patterns from data to make intelligent decisions or predictions.


b) Use gemini to simulate 20 data points in json format containing the following fields: first_name, last_name, phone_number, email, department, salary, title. See if you can prompt to direct the LLM output to have swedish names, phone numbers in swedish format (+46 731 29 52), departments (IT, HR, marketing, sales), reasonable salary (you might need to check some swedish statistics on salaries) and corresponding titles within these departments.

In [11]:
from google import genai
from dotenv import load_dotenv
import os

prompt = """
Simulate 20 unique data points in json format with the following fields: first_name, last_name, phone_number, email, department, monthly_salary, title.

I want swedish first names and lastnames.
Swedish format on phone number example: (+46 731 26 32), department example:(IT, HR, Marketing, Sales).
Reasonable monthly_salary based on swedish statistics on salaries based on department, and corresponding titles within the departments example(department: IT, title: Data Engineer)

Note that i want a wide range of department and title values, not only the examples.
Check https://allastudier.se/jobb-o-l%C3%B6n/ and validate what departments(Yrkeskategorier) exists, and what titles(role) exists within each department, and what mean-salary each title has.
Round monthly_salary into nearest 1000 so that for example 41052 is 41000 and 43912 is 44000
Use only swedish words for department and title.

Example output in json format:
    {
        "first_name": "Pontus",
        "last_name": "Ågren",
        "phone_number": "+46 731 26 32",
        "email": "pontus@mail.com",
        "department": "IT",
        "title": "Data Engineer",
        "monthly_salary": "42000"
    }
    
Give me only output in json format.
NOT markdown.
"""
load_dotenv()
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

response = client.models.generate_content(
    model="gemini-2.5-flash", contents=prompt
)
print(response.text)

[
    {
        "first_name": "Anna",
        "last_name": "Karlsson",
        "phone_number": "+46 701 23 45",
        "email": "anna.karlsson@example.se",
        "department": "IT- och teknik",
        "title": "Systemutvecklare",
        "monthly_salary": "49000"
    },
    {
        "first_name": "Erik",
        "last_name": "Andersson",
        "phone_number": "+46 702 34 56",
        "email": "erik.andersson@example.se",
        "department": "IT- och teknik",
        "title": "IT-konsult",
        "monthly_salary": "50000"
    },
    {
        "first_name": "Emma",
        "last_name": "Johansson",
        "phone_number": "+46 703 45 67",
        "email": "emma.johansson@example.se",
        "department": "IT- och teknik",
        "title": "IT-tekniker",
        "monthly_salary": "38000"
    },
    {
        "first_name": "Karl",
        "last_name": "Nilsson",
        "phone_number": "+46 704 56 78",
        "email": "karl.nilsson@example.se",
        "department": "IT- och te

c) Now use pydantic to validate this json and put in proper schema that the fields should follow. You might need to do some processing such as removing backticks and maybe loading json data into a list with `json.loads()`. Also make sure that only correctly validated data should be stored.

In [ ]:
from pydantic import BaseModel, EmailStr, Field, computed_field, ValidationError
from typing import List
import json

data = json.loads(response.text)

class Employee(BaseModel):
    first_name: str
    last_name: str
    phone_number: str
    email: EmailStr
    department: str
    title: str
    monthly_salary: int
    

class EmployeeListResponse(BaseModel):
    results: List[Employee]


employees = []
for emp in data:
    try:
        employees.append(Employee.model_validate(emp))
    except ValidationError as err:
        print(f"Skipping employee: {emp} | Error: {err}")

employee_response = EmployeeListResponse(results=employees)

print(employee_response.model_dump())

{'results': [{'first_name': 'Anna', 'last_name': 'Karlsson', 'phone_number': '+46 701 23 45', 'email': 'anna.karlsson@example.se', 'department': 'IT- och teknik', 'title': 'Systemutvecklare', 'monthly_salary': 49000}, {'first_name': 'Erik', 'last_name': 'Andersson', 'phone_number': '+46 702 34 56', 'email': 'erik.andersson@example.se', 'department': 'IT- och teknik', 'title': 'IT-konsult', 'monthly_salary': 50000}, {'first_name': 'Emma', 'last_name': 'Johansson', 'phone_number': '+46 703 45 67', 'email': 'emma.johansson@example.se', 'department': 'IT- och teknik', 'title': 'IT-tekniker', 'monthly_salary': 38000}, {'first_name': 'Karl', 'last_name': 'Nilsson', 'phone_number': '+46 704 56 78', 'email': 'karl.nilsson@example.se', 'department': 'IT- och teknik', 'title': 'UX-designer', 'monthly_salary': 44000}, {'first_name': 'Sara', 'last_name': 'Eriksson', 'phone_number': '+46 705 67 89', 'email': 'sara.eriksson@example.se', 'department': 'Ekonomi, juridik och administration', 'title': '

d) Write this json data to a folder called output_data.

In [25]:
os.makedirs("output_data", exist_ok=True)

with open("output_data/employee_response.json", "w", encoding="utf-8") as file:
    file.write(employee_response.model_dump_json(indent=2))

e) Use pandas to read the data as dataframe

In [ ]:
import pandas as pd

df = pd.read_json("output_data/employee_response.json", encoding="utf-8")
df.head()

In [34]:
with open("output_data/employee_response.json", "r", encoding="utf-8") as file:
    data = json.load(file)

results = data["results"]
df = pd.DataFrame(results)
df.head()

,first_name,last_name,phone_number,email,department,title,monthly_salary
0,Anna,Karlsson,+46 701 23 45,anna.karlsson@example.se,IT- och teknik,Systemutvecklare,49000
1,Erik,Andersson,+46 702 34 56,erik.andersson@example.se,IT- och teknik,IT-konsult,50000
2,Emma,Johansson,+46 703 45 67,emma.johansson@example.se,IT- och teknik,IT-tekniker,38000
3,Karl,Nilsson,+46 704 56 78,karl.nilsson@example.se,IT- och teknik,UX-designer,44000
4,Sara,Eriksson,+46 705 67 89,sara.eriksson@example.se,"Ekonomi, juridik och administration",Ekonom,43000


f) Write a csv file to your output_data

In [35]:
df.to_csv("output_data/employee_response.csv")

g) Load this data into a staging layer and store this into a table called employees.

In [39]:
import duckdb as db

with db.connect("company.duckdb") as conn:
    conn.execute("CREATE SCHEMA IF NOT EXISTS staging")
    conn.execute("CREATE TABLE IF NOT EXISTS staging.employees AS SELECT * FROM df")

In [40]:
df["department"].unique()

array(['IT- och teknik', 'Ekonomi, juridik och administration',
       'Marknadsföring, media och design', 'Sälj, inköp och logistik',
       'Vård och omsorg', 'Bygg och anläggning',
       'Utbildning och forskning'], dtype=object)

h) Use gemini to simulate departments data. There should be same departments as those you had in task b. Also add a description field and a contact person.

In [ ]:
prompt = f"""
Simulate data for the following departments: {df["department"].unique()}. with the following fields: department, description, contact_person.

The data should contain the department name, a description for the department, and a contact person for the department.

For the description and department_name i want you to use swedish words and sentences.
For contact_person i want the full name (first_name + last_name) from the person with the highest monthly_salary for each department: {db.query("SELECT first_name, last_name, department, monthly_salary FROM df")}

Example output in json-format:
    {{
        "department": "IT- och teknik"
        "description": "(ge en tydlig och kortfattad beskrivning av vad IT- och teknik innebär)"
        "contact_person": "Erik Andersson"
    }}

Give me only output in json format.
NOT markdown.

"""

load_dotenv()
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

response = client.models.generate_content(
    model="gemini-2.5-flash", contents=prompt
)
print(response.text)

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'The model is overloaded. Please try again later.', 'status': 'UNAVAILABLE'}}

In [46]:
df["department"].nunique()

7

In [45]:
print(response.text)

[
    {
        "department": "IT- och teknik",
        "description": "Ansvarar för att utveckla, implementera och underhålla företagets IT-system, nätverk och tekniska infrastruktur för att säkerställa effektivitet och innovation.",
        "contact_person": "Erik Andersson"
    },
    {
        "department": "Ekonomi, juridik och administration",
        "description": "Hantera företagets finansiella planering, redovisning, juridiska frågor samt övergripande administrativa processer för att stödja organisationens drift och efterlevnad.",
        "contact_person": "Maria Olsson"
    },
    {
        "department": "Marknadsföring, media och design",
        "description": "Skapa och implementera strategier för varumärkesbyggande, kommunikation och produktmarknadsföring genom olika kanaler, inklusive digital media och grafisk design.",
        "contact_person": "Linnea Pettersson"
    },
    {
        "department": "Sälj, inköp och logistik",
        "description": "Fokusera på försälj

In [51]:
unique_departments = df["department"].unique().tolist()
unique_departments

['IT- och teknik',
 'Ekonomi, juridik och administration',
 'Marknadsföring, media och design',
 'Sälj, inköp och logistik',
 'Vård och omsorg',
 'Bygg och anläggning',
 'Utbildning och forskning']

In [53]:
unique_employee_names = (df["first_name"] + " " + df["last_name"]).tolist()
unique_employee_names

['Anna Karlsson',
 'Erik Andersson',
 'Emma Johansson',
 'Karl Nilsson',
 'Sara Eriksson',
 'Johan Larsson',
 'Maria Olsson',
 'Gustav Persson',
 'Sofia Svensson',
 'Viktor Gustafsson',
 'Linnea Pettersson',
 'Fredrik Jonsson',
 'Elin Jansson',
 'David Bengtsson',
 'Lena Lindberg',
 'Oskar Henriksson',
 'Jenny Lindgren',
 'Anders Bergström',
 'Malin Lundberg',
 'Per Holm']

In [55]:
contact_to_department = dict(zip(df["first_name"] + " " + df["last_name"], df["department"]))
contact_to_department

{'Anna Karlsson': 'IT- och teknik',
 'Erik Andersson': 'IT- och teknik',
 'Emma Johansson': 'IT- och teknik',
 'Karl Nilsson': 'IT- och teknik',
 'Sara Eriksson': 'Ekonomi, juridik och administration',
 'Johan Larsson': 'Ekonomi, juridik och administration',
 'Maria Olsson': 'Ekonomi, juridik och administration',
 'Gustav Persson': 'Ekonomi, juridik och administration',
 'Sofia Svensson': 'Marknadsföring, media och design',
 'Viktor Gustafsson': 'Marknadsföring, media och design',
 'Linnea Pettersson': 'Marknadsföring, media och design',
 'Fredrik Jonsson': 'Sälj, inköp och logistik',
 'Elin Jansson': 'Sälj, inköp och logistik',
 'David Bengtsson': 'Sälj, inköp och logistik',
 'Lena Lindberg': 'Vård och omsorg',
 'Oskar Henriksson': 'Vård och omsorg',
 'Jenny Lindgren': 'Bygg och anläggning',
 'Anders Bergström': 'Bygg och anläggning',
 'Malin Lundberg': 'Utbildning och forskning',
 'Per Holm': 'IT- och teknik'}

In [57]:
highest_salary_dept = df.loc[df.groupby("department")["monthly_salary"].idxmax()]
highest_salary_dept

,first_name,last_name,phone_number,email,department,title,monthly_salary
16,Jenny,Lindgren,+46 717 89 01,jenny.lindgren@example.se,Bygg och anläggning,Byggnadsingenjör,45000
6,Maria,Olsson,+46 707 89 01,maria.olsson@example.se,"Ekonomi, juridik och administration",Jurist,51000
1,Erik,Andersson,+46 702 34 56,erik.andersson@example.se,IT- och teknik,IT-konsult,50000
10,Linnea,Pettersson,+46 711 23 45,linnea.pettersson@example.se,"Marknadsföring, media och design",Kommunikatör,39000
12,Elin,Jansson,+46 713 45 67,elin.jansson@example.se,"Sälj, inköp och logistik",Inköpare,39000
18,Malin,Lundberg,+46 719 01 23,malin.lundberg@example.se,Utbildning och forskning,Grundskollärare,38000
15,Oskar,Henriksson,+46 716 78 90,oskar.henriksson@example.se,Vård och omsorg,Psykolog,47000


In [58]:
highest_salary_contact = dict(zip(highest_salary_dept["first_name"] + " " + highest_salary_dept["last_name"], highest_salary_dept["monthly_salary"]))
highest_salary_contact

{'Jenny Lindgren': 45000,
 'Maria Olsson': 51000,
 'Erik Andersson': 50000,
 'Linnea Pettersson': 39000,
 'Elin Jansson': 39000,
 'Malin Lundberg': 38000,
 'Oskar Henriksson': 47000}

In [60]:
highest_salary_dept = df.loc[df.groupby("department")["monthly_salary"].idxmax()]
contact_to_department = dict(zip(highest_salary_dept["first_name"] + " " + highest_salary_dept["last_name"], highest_salary_dept["department"]))
contact_to_department

{'Jenny Lindgren': 'Bygg och anläggning',
 'Maria Olsson': 'Ekonomi, juridik och administration',
 'Erik Andersson': 'IT- och teknik',
 'Linnea Pettersson': 'Marknadsföring, media och design',
 'Elin Jansson': 'Sälj, inköp och logistik',
 'Malin Lundberg': 'Utbildning och forskning',
 'Oskar Henriksson': 'Vård och omsorg'}

i) Add a departments table in your duckdb database under staging layer to store this data.

In [63]:
from pydantic import model_validator

unique_departments = df["department"].unique().tolist()
unique_employee_names = (df["first_name"] + " " + df["last_name"]).tolist()

highest_salary_dept = df.loc[df.groupby("department")["monthly_salary"].idxmax()]
contact_to_department = dict(zip(highest_salary_dept["first_name"] + " " + highest_salary_dept["last_name"], highest_salary_dept["department"]))

class Department(BaseModel):
    department: str
    description: str
    contact_person: str
    
    
    @model_validator(mode="before")
    def validate_contact_to_department(cls, values):
        dep = values.get("department")
        contact = values.get("contact_person")
        
        # kontrollera att department existerar i andra tabellen
        if dep not in unique_departments:
            raise ValueError(f"'{values}' is not a valid department")
        
        # kontrollera att personen existerar i andra tabellen
        if contact not in unique_employee_names:
            raise ValueError(f"'{values}' is not a valid contact person")
        
        # kontrollera att personen är den högst betalda personen för sin department och då alltså dess kontakt person
        if contact not in contact_to_department:
            raise ValueError(f"'{contact}' is an employee, but is not the contact person for '{dep}'")

        # kontrollera att personen hör till rätt department
        correct_dep = contact_to_department.get(contact)
        if correct_dep != dep:
            raise ValueError(f"Contact person '{contact}' does not belong to department '{dep}'")
        
        return values


class DepartmentListResponse(BaseModel):
    results: List[Department]

data = json.loads(response.text)

departments = []
for dep in data:
    try:
        departments.append(Department.model_validate(dep))
    except ValidationError as err:
        print(f"Skipping department: {dep} | Error: {err}")

departments_response = DepartmentListResponse(results=departments)

print(departments_response.model_dump())  

{'results': [{'department': 'IT- och teknik', 'description': 'Ansvarar för att utveckla, implementera och underhålla företagets IT-system, nätverk och tekniska infrastruktur för att säkerställa effektivitet och innovation.', 'contact_person': 'Erik Andersson'}, {'department': 'Ekonomi, juridik och administration', 'description': 'Hantera företagets finansiella planering, redovisning, juridiska frågor samt övergripande administrativa processer för att stödja organisationens drift och efterlevnad.', 'contact_person': 'Maria Olsson'}, {'department': 'Marknadsföring, media och design', 'description': 'Skapa och implementera strategier för varumärkesbyggande, kommunikation och produktmarknadsföring genom olika kanaler, inklusive digital media och grafisk design.', 'contact_person': 'Linnea Pettersson'}, {'department': 'Sälj, inköp och logistik', 'description': 'Fokusera på försäljning av produkter eller tjänster, hantering av inköp från leverantörer samt optimering av logistikflöden för att

In [65]:
with open("output_data/department_response.json", "w", encoding="utf-8") as file:
    file.write(departments_response.model_dump_json(indent=2))

In [66]:
with open("output_data/department_response.json", "r", encoding="utf-8") as file:
    data = json.load(file)

results = data["results"]
df = pd.DataFrame(results)
df.head()

,department,description,contact_person
0,IT- och teknik,"Ansvarar för att utveckla, implementera och un...",Erik Andersson
1,"Ekonomi, juridik och administration","Hantera företagets finansiella planering, redo...",Maria Olsson
2,"Marknadsföring, media och design",Skapa och implementera strategier för varumärk...,Linnea Pettersson
3,"Sälj, inköp och logistik",Fokusera på försäljning av produkter eller tjä...,Elin Jansson
4,Vård och omsorg,Erbjuda professionell vård och stöd till indiv...,Oskar Henriksson


In [67]:
df.to_csv("output_data/department_response.csv")

In [68]:
with db.connect("company.duckdb") as conn:
    conn.execute("CREATE SCHEMA IF NOT EXISTS staging")
    conn.execute("CREATE TABLE IF NOT EXISTS staging.departments AS SELECT * FROM df")